# Text Halo

A text halo draws an outline around text, improving readability when labels are placed on top of colorful, textured, or busy plot areas.

Use `halo_width` to set the halo width and `halo_color` to set the halo color. When `halo_color` is omitted, the theme paper (fill) color is used.

In [1]:
import geopandas as gpd
import numpy as np
import pandas as pd
from lets_plot import *
from lets_plot.bistro.corr import corr_plot

LetsPlot.setup_html()

ModuleNotFoundError: No module named 'geopandas'

In [2]:
mpg = (
    pd.read_csv('https://raw.githubusercontent.com/JetBrains/lets-plot-docs/master/data/mpg.csv')
    .drop(columns=['Unnamed: 0'])
)

mpg_num = mpg.select_dtypes(include=np.number)
mpg_num.head(3)

,displ,year,cyl,cty,hwy
0,1.8,1999,4,18,29
1,1.8,1999,4,21,29
2,2.0,2008,4,20,31


## Correlation Plot Labels

A dark halo keeps white labels readable over tiles with different fill colors.

In [ ]:
gggrid([
    corr_plot(mpg_num)
    .tiles()
    .labels(color='white')
    .build(),

    corr_plot(mpg_num)
    .tiles()
    .labels(color='white', halo_width=1.6, halo_color='gray40')
    .build()
])

## Compare Text Labels With and Without Halo

The same scatter plot can be reused to compare plain text labels with labels that have a halo. The halo helps preserve the label color while separating the text from nearby points.

In [ ]:
sample = (
    mpg.sort_values('hwy', ascending=False)
    .groupby('manufacturer', as_index=False)
    .head(1)
)

base_plot = (
    ggplot(mpg, aes('displ', 'hwy'))
    + geom_point(aes(color='class'), size=4, alpha=.75)
    + theme_minimal()
    + ggsize(520, 360)
)

label_options = dict(
    data=sample,
    color='gray10',
    size=7,
    check_overlap=True,
)

no_halo = (
    base_plot
    + geom_text(aes(label='manufacturer'), **label_options)
    + ggtitle('Labels without halo')
)

with_halo = (
    base_plot
    + geom_text(aes(label='manufacturer'), halo_width=1.5, halo_color='white', **label_options)
    + ggtitle('Labels with white halo')
)

gggrid([no_halo, with_halo], ncol=1)

### High-Contrast Dark Flavor

With a dark theme, omitting `halo_color` automatically uses the theme paper color as the halo so the text stays legible without manual color tuning.

In [ ]:
dark_base = base_plot + flavor_high_contrast_dark()

no_halo_dark = (
    dark_base
    + geom_text(aes(label='manufacturer'), **label_options)
    + ggtitle('Dark theme — no halo')
)

with_halo_dark = (
    dark_base
    + geom_text(aes(label='manufacturer'), halo_width=1.5, **label_options)
    + ggtitle('Dark theme — theme-derived halo color')
)

gggrid([no_halo_dark, with_halo_dark], ncol=1)

## Repelled Labels on a Map

For geographic labels, a light halo separates place names from map outlines and neighboring markers while keeping the map styling simple.

In [ ]:
base_url = 'https://raw.githubusercontent.com/JetBrains/lets-plot-kotlin/master/docs/examples/shp/'
world_gdf = gpd.read_file(base_url + 'naturalearth_lowres/naturalearth_lowres.shp')
cities_gdf = gpd.read_file(base_url + 'naturalearth_cities/naturalearth_cities.shp')

(
    ggplot()
    + geom_map(map=world_gdf, fill='light_green', alpha=.3)
    + geom_point(map=cities_gdf, color='dark_slate_gray', size=3)
    + geom_text_repel(
        aes(label='name'),
        data=cities_gdf,
        color='dark_slate_gray',
        seed=42,
        max_iter=200,
        halo_width=2.0,
        halo_color='white',
    )
    + coord_map(xlim=[-10.5, 44.0], ylim=[37.0, 60.5])
    + theme(axis='blank', panel_grid='blank')
    + ggsize(800, 600)
)